## Agenda
- load the data;
- filter out only transactions that happened in `2022 after June` to not overload my resources:)
- check for missing values and assign meaningful value like 'No Promo Code'
- explore how long is the list in `product_metadata` column in transactions table
- keep array values as they are because Postgres can handle array values and save the transformed csv files

In [1]:
import pandas as pd
import ast
import json

In [2]:
clicks = pd.read_csv("org_dataset/click_stream.csv")
transcations = pd.read_csv("org_dataset/transactions.csv")

In [3]:
def filter_2022_data(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df_transformed = df.copy()
    return df_transformed[
        (pd.to_datetime(df_transformed[date_col]).dt.year == 2022) 
        & (pd.to_datetime(df_transformed[date_col]).dt.month >= 6)
    ]

In [4]:
clicks_2022 = filter_2022_data(clicks, "event_time").reset_index(drop=True)
transcations_2022 = filter_2022_data(transcations, "created_at").reset_index(drop=True)

In [5]:
clicks_2022.isna().sum()

session_id             0
event_name             0
event_time             0
event_id               0
traffic_source         0
event_metadata    624200
dtype: int64

In [6]:
transcations_2022.isna().sum()

created_at                    0
customer_id                   0
booking_id                    0
session_id                    0
product_metadata              0
payment_method                0
payment_status                0
promo_amount                  0
promo_code                37801
shipment_fee                  0
shipment_date_limit           0
shipment_location_lat         0
shipment_location_long        0
total_amount                  0
dtype: int64

In [7]:
clicks_2022 = clicks_2022.fillna({"event_metadata":"No_Choice"})
transcations_2022 = transcations_2022.fillna({"promo_code":"No_Promo"})

In [8]:
clicks_2022.head()

,session_id,event_name,event_time,event_id,traffic_source,event_metadata
0,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,HOMEPAGE,2022-06-03T02:26:08.425431Z,83554a77-dbf6-4ff9-8540-a28f09532b83,MOBILE,No_Choice
1,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,ADD_TO_CART,2022-06-04T05:58:24.425431Z,38dbca3e-f1e7-42e6-85cf-ddcb99571fb5,MOBILE,"{'product_id': 36038, 'quantity': 1, 'item_pri..."
2,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,BOOKING,2022-06-06T12:48:42.425431Z,2e23a824-7fb9-4889-b8e6-b2f035f3efb4,MOBILE,{'payment_status': 'Success'}
3,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,HOMEPAGE,2022-06-04T06:00:07.425431Z,6d2cf796-b7f4-4891-85c3-1f0c1c3de1cd,MOBILE,No_Choice
4,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,PROMO_PAGE,2022-06-06T13:09:01.425431Z,ecc6427a-a654-431a-8032-93da3627511c,MOBILE,No_Choice


In [9]:
transcations_2022.head()

,created_at,customer_id,booking_id,session_id,product_metadata,payment_method,payment_status,promo_amount,promo_code,shipment_fee,shipment_date_limit,shipment_location_lat,shipment_location_long,total_amount
0,2022-06-09T12:18:09.365620Z,4774,7ab6eac3-ff92-46d7-9030-296301ee920a,1688279e-d03f-466a-a20e-57846de4c179,"[{'product_id': 18476, 'quantity': 1, 'item_pr...",Credit Card,Success,3329,XX2022,50000,2022-06-10T17:11:20.049154Z,0.148523,109.320337,245215
1,2022-06-08T18:16:44.162570Z,58191,18de3252-7d71-49d2-b7b2-b8c77f99b215,0563b55a-a859-45a3-920a-2a60cea60a32,"[{'product_id': 53744, 'quantity': 1, 'item_pr...",OVO,Success,0,No_Promo,15000,2022-06-14T16:54:44.526655Z,-7.127042,106.664946,539267
2,2022-06-23T18:02:28.162570Z,58191,45ba4e9c-bb8a-4adf-a7a0-48ab3faff235,488c9d1c-4fb4-4a42-b08d-1eb7f461e7d8,"[{'product_id': 9996, 'quantity': 1, 'item_pri...",OVO,Success,0,No_Promo,0,2022-06-25T03:38:08.828838Z,-8.328679,118.773797,229473
3,2022-07-08T18:02:01.162570Z,58191,e03c7345-b4f9-4d0d-b53c-b039f0b2fdb8,da19c53e-6ccf-40d9-bfe1-5bb1aea42680,"[{'product_id': 6973, 'quantity': 1, 'item_pri...",OVO,Success,9376,BUYMORE,10000,2022-07-12T20:12:47.620520Z,-2.897258,120.314504,171940
4,2022-06-12T09:55:06.614167Z,76966,7d30465e-1098-4e39-b6f7-7d6b3f829625,63406f02-ddfb-4816-a77f-efdd494b8abe,"[{'product_id': 18558, 'quantity': 1, 'item_pr...",Credit Card,Success,7597,WEEKENDSERU,0,2022-06-17T06:11:28.122018Z,-1.839813,114.001993,152828


In [10]:
transcations_2022["length"] = transcations_2022["product_metadata"].apply(ast.literal_eval).apply(len)

In [11]:
transcations_2022

,created_at,customer_id,booking_id,session_id,product_metadata,payment_method,payment_status,promo_amount,promo_code,shipment_fee,shipment_date_limit,shipment_location_lat,shipment_location_long,total_amount,length
0,2022-06-09T12:18:09.365620Z,4774,7ab6eac3-ff92-46d7-9030-296301ee920a,1688279e-d03f-466a-a20e-57846de4c179,"[{'product_id': 18476, 'quantity': 1, 'item_pr...",Credit Card,Success,3329,XX2022,50000,2022-06-10T17:11:20.049154Z,0.148523,109.320337,245215,1
1,2022-06-08T18:16:44.162570Z,58191,18de3252-7d71-49d2-b7b2-b8c77f99b215,0563b55a-a859-45a3-920a-2a60cea60a32,"[{'product_id': 53744, 'quantity': 1, 'item_pr...",OVO,Success,0,No_Promo,15000,2022-06-14T16:54:44.526655Z,-7.127042,106.664946,539267,1
2,2022-06-23T18:02:28.162570Z,58191,45ba4e9c-bb8a-4adf-a7a0-48ab3faff235,488c9d1c-4fb4-4a42-b08d-1eb7f461e7d8,"[{'product_id': 9996, 'quantity': 1, 'item_pri...",OVO,Success,0,No_Promo,0,2022-06-25T03:38:08.828838Z,-8.328679,118.773797,229473,1
3,2022-07-08T18:02:01.162570Z,58191,e03c7345-b4f9-4d0d-b53c-b039f0b2fdb8,da19c53e-6ccf-40d9-bfe1-5bb1aea42680,"[{'product_id': 6973, 'quantity': 1, 'item_pri...",OVO,Success,9376,BUYMORE,10000,2022-07-12T20:12:47.620520Z,-2.897258,120.314504,171940,1
4,2022-06-12T09:55:06.614167Z,76966,7d30465e-1098-4e39-b6f7-7d6b3f829625,63406f02-ddfb-4816-a77f-efdd494b8abe,"[{'product_id': 18558, 'quantity': 1, 'item_pr...",Credit Card,Success,7597,WEEKENDSERU,0,2022-06-17T06:11:28.122018Z,-1.839813,114.001993,152828,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53744,2022-06-27T08:10:37.025516Z,80629,8e2717a7-fa34-48b9-8bcd-42621d441692,354406ed-5356-4bd8-befa-0ae442b9e527,"[{'product_id': 36914, 'quantity': 5, 'item_pr...",Credit Card,Success,2788,AZ2022,10000,2022-07-02T01:17:34.274005Z,-4.675292,104.150059,1961390,7
53745,2022-06-04T07:42:42.672997Z,67945,8570fa7f-ce2b-48e4-ac15-b88294199dc3,f38cb266-5368-4718-bed1-e6eaf73670b9,"[{'product_id': 22179, 'quantity': 1, 'item_pr...",Gopay,Success,5336,WEEKENDSERU,5000,2022-06-06T00:01:03.843900Z,-7.753956,110.395514,161526,1
53746,2022-07-03T15:52:23.672997Z,67945,0e813793-6434-40db-92bb-78edbf75ee19,2f4c84a5-6df8-4cb8-9b79-3911c5caa272,"[{'product_id': 24158, 'quantity': 1, 'item_pr...",Gopay,Success,0,No_Promo,10000,2022-07-06T14:21:32.221351Z,-6.182888,106.750594,1523371,5
53747,2022-06-08T02:44:08.144627Z,99675,ca733563-1455-482d-bae0-e47337304019,8d71b0a4-e0f8-47ea-9de4-b11c0db924c1,"[{'product_id': 15331, 'quantity': 1, 'item_pr...",Credit Card,Success,0,No_Promo,0,2022-06-09T03:06:35.978567Z,-8.093308,110.709094,237077,1


In [12]:
transcations_2022.length.max()

43

In [13]:
transcations_2022.nlargest(10, "length")[["product_metadata", "length"]]

,product_metadata,length
50301,"[{'product_id': 25166, 'quantity': 1, 'item_pr...",43
20195,"[{'product_id': 57945, 'quantity': 1, 'item_pr...",39
17208,"[{'product_id': 10145, 'quantity': 1, 'item_pr...",36
39260,"[{'product_id': 17683, 'quantity': 1, 'item_pr...",29
33529,"[{'product_id': 25181, 'quantity': 1, 'item_pr...",28
10248,"[{'product_id': 56261, 'quantity': 1, 'item_pr...",27
37097,"[{'product_id': 9129, 'quantity': 1, 'item_pri...",27
48412,"[{'product_id': 29106, 'quantity': 1, 'item_pr...",27
10173,"[{'product_id': 25549, 'quantity': 1, 'item_pr...",26
15479,"[{'product_id': 43911, 'quantity': 13, 'item_p...",26


In [35]:
transcations_2022.iloc[50301, 0:4]

booking_id     1b77d403-c552-490f-a1e9-4f02634f56b0
session_id     5aa4bcd3-d4a7-400f-a11f-5cf38b751fe4
customer_id                                   66915
created_at              2022-07-14T19:10:41.302769Z
Name: 50301, dtype: object

In [14]:
transcations_2022.drop(columns=["length"], inplace=True)

In [15]:
def swap_columns(df, col1, col2):
    cols = df.columns.tolist()
    i, j = cols.index(col1), cols.index(col2)
    cols[i], cols[j] = cols[j], cols[i]
    return df[cols]

In [16]:
clicks_2022 = swap_columns(clicks_2022, "session_id", "event_id")
clicks_2022 = swap_columns(clicks_2022, "event_name", "session_id")
clicks_2022 = swap_columns(clicks_2022, "event_time", "event_name")

In [17]:
clicks_2022.head()

,event_id,session_id,event_name,event_time,traffic_source,event_metadata
0,83554a77-dbf6-4ff9-8540-a28f09532b83,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,HOMEPAGE,2022-06-03T02:26:08.425431Z,MOBILE,No_Choice
1,38dbca3e-f1e7-42e6-85cf-ddcb99571fb5,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,ADD_TO_CART,2022-06-04T05:58:24.425431Z,MOBILE,"{'product_id': 36038, 'quantity': 1, 'item_pri..."
2,2e23a824-7fb9-4889-b8e6-b2f035f3efb4,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,BOOKING,2022-06-06T12:48:42.425431Z,MOBILE,{'payment_status': 'Success'}
3,6d2cf796-b7f4-4891-85c3-1f0c1c3de1cd,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,HOMEPAGE,2022-06-04T06:00:07.425431Z,MOBILE,No_Choice
4,ecc6427a-a654-431a-8032-93da3627511c,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,PROMO_PAGE,2022-06-06T13:09:01.425431Z,MOBILE,No_Choice


In [18]:
transcations_2022 = swap_columns(transcations_2022, "created_at", "booking_id")
transcations_2022 = swap_columns(transcations_2022, "customer_id", "session_id")
transcations_2022 = swap_columns(transcations_2022, "created_at", "customer_id")

In [19]:
transcations_2022.head()

,booking_id,session_id,customer_id,created_at,product_metadata,payment_method,payment_status,promo_amount,promo_code,shipment_fee,shipment_date_limit,shipment_location_lat,shipment_location_long,total_amount
0,7ab6eac3-ff92-46d7-9030-296301ee920a,1688279e-d03f-466a-a20e-57846de4c179,4774,2022-06-09T12:18:09.365620Z,"[{'product_id': 18476, 'quantity': 1, 'item_pr...",Credit Card,Success,3329,XX2022,50000,2022-06-10T17:11:20.049154Z,0.148523,109.320337,245215
1,18de3252-7d71-49d2-b7b2-b8c77f99b215,0563b55a-a859-45a3-920a-2a60cea60a32,58191,2022-06-08T18:16:44.162570Z,"[{'product_id': 53744, 'quantity': 1, 'item_pr...",OVO,Success,0,No_Promo,15000,2022-06-14T16:54:44.526655Z,-7.127042,106.664946,539267
2,45ba4e9c-bb8a-4adf-a7a0-48ab3faff235,488c9d1c-4fb4-4a42-b08d-1eb7f461e7d8,58191,2022-06-23T18:02:28.162570Z,"[{'product_id': 9996, 'quantity': 1, 'item_pri...",OVO,Success,0,No_Promo,0,2022-06-25T03:38:08.828838Z,-8.328679,118.773797,229473
3,e03c7345-b4f9-4d0d-b53c-b039f0b2fdb8,da19c53e-6ccf-40d9-bfe1-5bb1aea42680,58191,2022-07-08T18:02:01.162570Z,"[{'product_id': 6973, 'quantity': 1, 'item_pri...",OVO,Success,9376,BUYMORE,10000,2022-07-12T20:12:47.620520Z,-2.897258,120.314504,171940
4,7d30465e-1098-4e39-b6f7-7d6b3f829625,63406f02-ddfb-4816-a77f-efdd494b8abe,76966,2022-06-12T09:55:06.614167Z,"[{'product_id': 18558, 'quantity': 1, 'item_pr...",Credit Card,Success,7597,WEEKENDSERU,0,2022-06-17T06:11:28.122018Z,-1.839813,114.001993,152828


In [20]:
clicks_2022["event_metadata"].value_counts()

event_metadata
No_Choice                                                     624200
{'payment_status': 'Success'}                                  51475
{'search_keywords': 'Dress Kondangan'}                         19775
{'search_keywords': 'Tas Wanita'}                               9900
{'search_keywords': 'Bekas'}                                    8764
                                                               ...  
{'product_id': 19914, 'quantity': 1, 'item_price': 189261}         1
{'promo_code': 'SC2022', 'promo_amount': 12687}                    1
{'product_id': 10503, 'quantity': 1, 'item_price': 96025}          1
{'product_id': 34581, 'quantity': 1, 'item_price': 73132}          1
{'product_id': 41607, 'quantity': 3, 'item_price': 212381}         1
Name: count, Length: 97959, dtype: int64

### Converting `event_metadata` into correct json format to insert into table as `jsonb` type

In [21]:
def to_json(x):
    if pd.isna(x) or x == "No_Choice":
        return "{}"

    try:
        return json.dumps(ast.literal_eval(x))
    except:
        return "{}"

In [22]:
clicks_2022["event_metadata"] = clicks_2022["event_metadata"].apply(to_json)

In [27]:
clicks_2022.head()

,event_id,session_id,event_name,event_time,traffic_source,event_metadata
0,83554a77-dbf6-4ff9-8540-a28f09532b83,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,HOMEPAGE,2022-06-03T02:26:08.425431Z,MOBILE,{}
1,38dbca3e-f1e7-42e6-85cf-ddcb99571fb5,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,ADD_TO_CART,2022-06-04T05:58:24.425431Z,MOBILE,"{""product_id"": 36038, ""quantity"": 1, ""item_pri..."
2,2e23a824-7fb9-4889-b8e6-b2f035f3efb4,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,BOOKING,2022-06-06T12:48:42.425431Z,MOBILE,"{""payment_status"": ""Success""}"
3,6d2cf796-b7f4-4891-85c3-1f0c1c3de1cd,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,HOMEPAGE,2022-06-04T06:00:07.425431Z,MOBILE,{}
4,ecc6427a-a654-431a-8032-93da3627511c,6968fd4a-2ad4-44a4-9a3b-fc1b8ae31ebd,PROMO_PAGE,2022-06-06T13:09:01.425431Z,MOBILE,{}


### Converting `product_metadata` into correct json format to insert into table as `jsonb` type

In [23]:
def from_list_to_json(x):
    if pd.isna(x):
        return "[]"

    try:
        return json.dumps(ast.literal_eval(x))
    except:
        return "[]"

In [24]:
transcations_2022["product_metadata"] = transcations_2022["product_metadata"].apply(to_json)

In [25]:
transcations_2022["product_metadata"].value_counts()

product_metadata
[{"product_id": 18476, "quantity": 1, "item_price": 198544}]    1
[{"product_id": 4689, "quantity": 1, "item_price": 252066}]     1
[{"product_id": 46278, "quantity": 1, "item_price": 247164}]    1
[{"product_id": 3967, "quantity": 1, "item_price": 82256}]      1
[{"product_id": 4800, "quantity": 1, "item_price": 289415}]     1
                                                               ..
[{"product_id": 48767, "quantity": 1, "item_price": 180889}]    1
[{"product_id": 58330, "quantity": 2, "item_price": 235963}]    1
[{"product_id": 23533, "quantity": 1, "item_price": 270399}]    1
[{"product_id": 45633, "quantity": 1, "item_price": 427327}]    1
[{"product_id": 35318, "quantity": 1, "item_price": 488722}]    1
Name: count, Length: 53749, dtype: int64

In [26]:
transcations_2022.to_csv("dataset/transactions.csv", index=False)
clicks_2022.to_csv("dataset/clicks.csv", index=False)